# Formula-SWCT Generalization on Imagenette

This notebook evaluates the final Formula-SWCT method on Imagenette using an ImageNet-pretrained ResNet-18.

Unlike Beans, DTD, and Flowers102, Imagenette is not used to define the final fixed-beta calibration targets.

Formula-SWCT follows a predict-first, train-once protocol:

1. Collect stage statistics from the target training split using the frozen pretrained network.
2. Predict four stage-specific beta values using the previously calibrated Formula-SWCT predictor.
3. Freeze the predicted beta values.
4. Extract fixed train, validation, and test features once.
5. Train one downstream linear classifier.
6. Select the classifier checkpoint using validation accuracy.
7. Evaluate once on the test split.

No Imagenette-specific beta sweep or beta optimization is used.

Imagenette was used during method development and should therefore be interpreted as development/generalization evidence rather than as a completely untouched external validation dataset.

In [1]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

Tue Sep 29 19:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Environment Setup

The required research dependencies are installed below.

After installation, restart the Colab runtime before continuing. Do not rerun the installation cell after the restart.

In [2]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 57.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 3.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 123.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 134.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 15.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

### Runtime Restart Required

Restart the Colab runtime after the dependency installation.

After restarting, continue from the next cell rather than rerunning the installation commands.

In [3]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning


In [1]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


## Imagenette Dataset Setup

The Imagenette train, validation, and test loaders are constructed using the same transfer-learning pipeline used throughout the project.

This cell verifies the dataset sizes and batch structure before any evaluation is performed.

In [4]:
%cd /content/curvature-tuning-research/src/curvature-tuning/

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_imagenette",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/content/curvature-tuning-research/src/curvature-tuning


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

imagenette.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 7576
Validation samples: 1893
Test samples: 3925
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label min: 0
Label max: 9


# Baseline and Original S-CT Reference

The original Baseline and Single-Parameter Curvature Tuning methods are evaluated on Imagenette to provide a reference for Formula-SWCT.

S-CT performs its original validation-based search over a single global beta value.

For seed 42, the surviving experiment produced:

- Baseline test accuracy: **98.60%**
- S-CT test accuracy: **98.78%**
- Selected global beta: **0.94**
- Best validation accuracy: **99.00%**

Formula-SWCT is compared against these results while avoiding target-dataset beta search.

In [ ]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-27 00:55:25.686 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_imagenette_resnet18_seed42.log
2026-09-27 00:55:25.702 | INFO     | __main__:main:58 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 164MB/s]
README.md: 2.05kB [00:00, 8.48MB/s]
imagenette.py: 2.48kB [00:00, 10.5MB/s]
Generating train split: 9469 examples [00:54, 172.18 examples/s]
Generating test split: 3925 examples [00:22, 175.80 examples/s]
2026-09-27 00:58:42.258 | INFO     | __main__:main:85 - Testing baseline...
2026-09-27 00:58:42.280 | INFO     | __main__:main:88 - Number of trainable parameters: 5130
2026-09-27 00:58:42.280 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-27 00:59:34.547 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.861894, Accuracy: 0.00%
2026-09-27 00:59:34.830 | INFO     | train:train_epo

# Reconstructing the Frozen Formula-SWCT Predictor

Formula-SWCT is calibrated using only the three development datasets:

- Beans
- DTD
- Flowers102

The final fixed-beta calibration targets are:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

The predictor uses five frozen-network descriptors per stage and ridge regularization with lambda = 0.10.

This fitting step does not use Imagenette labels for beta optimization and does not use the Imagenette validation or test set to select beta values.

Once generated, the formula is frozen before the Imagenette evaluation.

In [5]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: FIT DIRECT BETA FORMULA
No beta sweep and no downstream training are performed here.

data/train-00000-of-00001.parquet: 100% 144M/144M [00:01<00:00, 80.9MB/s] 
data/validation-00000-of-00001.parquet: 100% 18.5M/18.5M [00:01<00:00, 16.4MB/s]
data/test-00000-of-00001.parquet: 100% 17.7M/17.7M [00:00<00:00, 24.2MB/s]
Generating train split: 1034 examples [00:00, 1613.80 examples/s]
Generating validation split: 133 examples [00:00, 2907.78 examples/s]
Generating test split: 128 examples [00:00, 3133.91 examples/s]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 125MB/s] 



100% 625M/625M [00:33<00:00, 18.8MB/s] 



100% 345M/345M [00:13<00:00, 25.7MB/s] 
100% 502/502 [00:00<00:00, 1.91MB/s]
100% 15.0k/15.0k [00:00<00:00, 61.6MB/s]



FITTED FORMULA
beta_hat = clip(b0 + sum(w_j * z_j), 0.7, 0.99)
intercept

# Final Formula-SWCT Evaluation

The calibrated Formula-SWCT predictor is now applied to Imagenette.

The four beta values are predicted from Imagenette training-split stage statistics before downstream classifier training.

The predicted beta values are then frozen, features are extracted once, and one downstream linear classifier is trained.

No beta sweep or beta optimization is performed.

In [6]:
!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds imagenette \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: imagenette
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.916325
Stage 2: 0.907159
Stage 3: 0.919142
Stage 4: 0.916368
Predicted beta vector: [0.916325, 0.907159, 0.919142, 0.916368]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.916325, 0.907159, 0.919142, 0.916368]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.187880 | val_acc=97.57%
Epoch 02 | val_loss=0.093569 | val_acc=98.31%
Epoch 03 | val_loss=0.076413 | val_acc=98.31%
Epoch 04 | val_loss=0.063843 | val_acc=98.47%
Epoch 05 | val_loss=0.059257 | val_acc=98.52%
Epoch 06 | val_loss=0.051689 | val_acc=98.52%
Epoch 07 | val_loss=0.051810 | val_acc=98.52%
Epoch 08 | val_loss=0.046792 | val_acc=98.68%
Epoch 09 | val_loss=0.048465 | val_acc=98.47%
Epoch 10 | val_loss=0.044072 | val_acc=98.52%
Epoch 11 | val_loss=0.043710 | val_acc=98.68%
Epoch 12 | val_

# Imagenette Results Summary

For seed 42:

| Method | Beta Selection | Test Accuracy |
|---|---|---:|
| Baseline | Original ReLU | **98.60%** |
| S-CT | Validation search over one global beta | **98.78%** |
| Formula-SWCT | Direct four-stage prediction | **98.78%** |

The final Formula-SWCT prediction is:

**[0.916325, 0.907159, 0.919142, 0.916368]**

with:

- Best validation accuracy: **98.84%**
- Best validation epoch: **12**
- Test accuracy: **98.78%**

Formula-SWCT therefore matches the S-CT test accuracy on this Imagenette run while eliminating the target-dataset beta search.

This result is treated as development/generalization evidence because Imagenette was used during the broader Formula-SWCT development process rather than as a completely untouched external validation dataset.